In [14]:
# Core libs
import numpy as np
import pandas as pd
import joblib


# Modeling
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# XGBoost
from xgboost import XGBClassifier # type: ignore

from sklearn.model_selection import cross_val_score, StratifiedKFold

In [15]:
df = pd.read_csv("cleve_heart.csv")
X_cols = ['Age', 'sex', 'chest pain type', 'Trestbps', 'cholesteral', 'fasting blood sugar',
          'resting ecg', 'max heart rate', 'exercise induced angina', 'oldpeak', 'slope',
          'number of vessels colored', 'thal']
y_col = 'healthy'


In [16]:
df.replace('?', np.nan, inplace=True)
df.replace('sick', 1, inplace=True)
df.replace('buff', 0, inplace=True)

C:\Users\krish\AppData\Local\Temp\ipykernel_8312\3327809629.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('buff', 0, inplace=True)


In [17]:
X_raw = df[X_cols].copy()
y = df['healthy'].values

In [18]:
categorical_cols = ['sex', 'chest pain type', 'Trestbps', 'resting ecg', 
                    'fasting blood sugar', 'exercise induced angina', 
                    'slope', 'thal']

numeric_cols = ['cholesteral', 'max heart rate', 
                'oldpeak', 'number of vessels colored']

In [19]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ], remainder="drop"
)

In [20]:
rf = RandomForestClassifier(
    n_estimators=400, n_jobs=-1, random_state=42
)

xgb = XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, objective="binary:logistic",
    eval_metric="logloss", n_jobs=-1, random_state=42, tree_method="hist"
)

meta_learner = LogisticRegression(max_iter=500, random_state=42, solver="lbfgs")

stack = StackingClassifier(
    estimators=[("rf", rf), ("xgb", xgb)],
    final_estimator=meta_learner,
    stack_method="predict_proba",
    passthrough=False,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# End-to-end pipeline with preprocessing
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("stack", stack)
])

In [ ]:
# clf = CalibratedClassifierCV(estimator=model, method="isotonic", cv=3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Fit
model.fit(X_train, y_train)
# joblib.dump(model, "heart_stack_calibrated2.pkl")
y_proba = model.predict_proba(X_test)[:, 1]


In [23]:
threshold = 0.5
y_pred = (y_proba >= threshold).astype(int)

# Evaluation metrics
auroc = roc_auc_score(y_test, y_proba)
auprc = average_precision_score(y_test, y_proba)

# Confusion matrix at chosen threshold
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Calibration curve data (for plotting later)
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy="quantile")

# ROC curve points
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)

# Precision-Recall curve points
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)

report = classification_report(y_test, y_pred)
# Print summary
print(report)
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")
print("Confusion Matrix (threshold=0.5):")
print(cm)
print(f"TPR (Recall): {tp / (tp + fn + 1e-12):.4f}")
print(f"FPR: {fp / (fp + tn + 1e-12):.4f}")


              precision    recall  f1-score   support

           0       0.80      0.85      0.82        33
           1       0.81      0.75      0.78        28

    accuracy                           0.80        61
   macro avg       0.80      0.80      0.80        61
weighted avg       0.80      0.80      0.80        61

AUROC: 0.8718
AUPRC: 0.8590
Confusion Matrix (threshold=0.5):
[[28  5]
 [ 7 21]]
TPR (Recall): 0.7500
FPR: 0.1515


In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_auroc = cross_val_score(
    model, X_raw, y, scoring="roc_auc", cv=cv_strategy, n_jobs=-1
)

# Cross-validated AUPRC
cv_scores_auprc = cross_val_score(
    model, X_raw, y, scoring="average_precision", cv=cv_strategy, n_jobs=-1
)

print(f"Cross-validated AUROC: {cv_scores_auroc.mean():.4f} ± {cv_scores_auroc.std():.4f}")
print(f"Cross-validated AUPRC: {cv_scores_auprc.mean():.4f} ± {cv_scores_auprc.std():.4f}")
print(f"All AUROC scores: {cv_scores_auroc}")
print(f"All AUPRC scores: {cv_scores_auprc}")


Cross-validated AUROC: 0.9009 ± 0.0222
Cross-validated AUPRC: 0.8913 ± 0.0365
All AUROC scores: [0.9047619  0.9237013  0.91125541 0.85858586 0.90628507]
All AUPRC scores: [0.92221612 0.92067667 0.92030556 0.84867385 0.84471438]


In [ ]:
joblib.dump(model, "heart_stack_calibrated.pkl")